In [9]:
import re
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

# Load CSV and remove missing rows
df = pd.read_csv("amazon-sampled.csv", sep=';', quotechar='"', on_bad_lines='skip')
df = df.dropna(subset=['Reviews', 'Rating'])

# Convert Rating into 3-class Sentiment
def get_sentiment(rating):
    rating = float(rating)
    if rating >= 4:
        return "positive"
    elif rating <= 2:
        return "negative"
    else:
        return "neutral"

df['Sentiment'] = df['Rating'].apply(get_sentiment)


stop_words = set(stopwords.words('english'))
negation_words = {'no', 'not', 'never'}
lemmatizer = WordNetLemmatizer()

def handle_negation(text):
    result = []
    negate = False
    for word in text:
        if word in negation_words:
            negate = True
            result.append(word)
        elif negate:
            result.append(f"NOT_{word}")
            if word in {'.', ',', '!', '?', ';', ':'}:  
                negate = False
        else:
            result.append(word)
    return result

def preprocess_text(text):
    tokenized = word_tokenize(text.lower())
    no_stopwords = [word for word in tokenized 
                    if word not in stop_words or word in negation_words]
    lemmatized = [lemmatizer.lemmatize(word) for word in no_stopwords]
    negation_handled = handle_negation(lemmatized)
    return negation_handled

df['cleaned'] = df['Reviews'].astype(str).apply(preprocess_text)


tfidf_vectorizer = TfidfVectorizer(tokenizer=lambda doc: doc, lowercase=False)
X = tfidf_vectorizer.fit_transform(df['cleaned'])  
y = df['Sentiment']                               

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# NB
nb = MultinomialNB().fit(X_train, y_train)
print("\n=== TF-IDF + Naive Bayes ===")
print("Accuracy:", accuracy_score(y_test, nb.predict(X_test)))
print(classification_report(y_test, nb.predict(X_test), digits=3))

# SVM
svm = LinearSVC(C=1.0).fit(X_train, y_train)
print("\n=== TF-IDF + SVM ===")
print("Accuracy:", accuracy_score(y_test, svm.predict(X_test)))
print(classification_report(y_test, svm.predict(X_test), digits=3))



=== TF-IDF + Naive Bayes ===
Accuracy: 0.7271589486858573
              precision    recall  f1-score   support

    negative      0.919     0.179     0.300       190
     neutral      0.000     0.000     0.000        60
    positive      0.718     0.996     0.834       549

    accuracy                          0.727       799
   macro avg      0.546     0.392     0.378       799
weighted avg      0.712     0.727     0.645       799


=== TF-IDF + SVM ===
Accuracy: 0.8385481852315394
              precision    recall  f1-score   support

    negative      0.743     0.805     0.773       190
     neutral      0.364     0.067     0.113        60
    positive      0.881     0.934     0.907       549

    accuracy                          0.839       799
   macro avg      0.663     0.602     0.598       799
weighted avg      0.810     0.839     0.816       799



/opt/anaconda3/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in label

In [19]:



# CountVectorizer fit 
count_vec = CountVectorizer(
    tokenizer=lambda doc: doc,
    preprocessor=lambda x: x,
    lowercase=False,  
)

X = count_vec.fit_transform(df['cleaned'])   


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# Train + evaluate Naive Bayes
nb = MultinomialNB(alpha=1.0)
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)
print("\n=== CountVectorizer + Naive Bayes ===")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb, digits=3))

#Train + evaluate SVM
svm = LinearSVC(C=1.0)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)
print("\n=== CountVectorizer + SVM ===")
print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm, digits=3))

/opt/anaconda3/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(



=== CountVectorizer + Naive Bayes ===
Accuracy: 0.8197747183979975
              precision    recall  f1-score   support

    negative      0.754     0.679     0.715       190
     neutral      0.000     0.000     0.000        60
    positive      0.846     0.958     0.898       549

    accuracy                          0.820       799
   macro avg      0.533     0.546     0.538       799
weighted avg      0.760     0.820     0.787       799


=== CountVectorizer + SVM ===
Accuracy: 0.8160200250312891
              precision    recall  f1-score   support

    negative      0.719     0.768     0.743       190
     neutral      0.182     0.100     0.129        60
    positive      0.888     0.911     0.899       549

    accuracy                          0.816       799
   macro avg      0.596     0.593     0.590       799
weighted avg      0.795     0.816     0.804       799



/opt/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:1242: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Sentiment
positive    2743
negative     950
neutral      300
Name: count, dtype: int64
Sentiment
positive    0.686952
negative    0.237916
neutral     0.075131
Name: proportion, dtype: float64
